In [1]:
import numpy as np
import pandas as pd
import OpenEXR
import re
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

In [2]:
df1 = pd.read_csv("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/notrees-dl.csv")
df2 = pd.read_csv("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/shadows-detection-dataset/trees-dl-shadows.csv")

In [3]:
df1["shadows"] = 0.0
df1

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut,shadows
0,0.187436,0.000161,0.000196,1124.690170,1020.232214,0.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.116122,0.0
1,0.092571,0.000277,0.000303,1124.690170,1020.232214,1.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.106717,0.0
2,0.057824,0.000337,0.000359,1124.690170,1020.232214,2.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.097078,0.0
3,0.171885,0.000224,0.000273,1124.690170,1020.232214,3.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.112769,0.0
4,0.110673,0.000253,0.000287,1124.690170,1020.232214,4.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.103993,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329615,0.222720,0.001223,0.001876,262.715686,200.963755,15.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.134427,0.0
329616,0.204124,0.001236,0.001864,262.715686,200.963755,16.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.132863,0.0
329617,0.231891,0.001070,0.001640,262.715686,200.963755,17.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.135519,0.0
329618,0.235853,0.000894,0.001359,262.715686,200.963755,18.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.135568,0.0


In [4]:
df2

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut,shadows
0,0.248512,0.000147,0.000204,1136.471,1031.228,0.0,180.0,-71.945084,19.489212,0.0,-19.350,-0.0,0.143482,0.0
1,0.209356,0.000254,0.000347,1136.471,1031.228,1.0,180.0,-71.945084,19.489212,0.0,-19.350,-0.0,0.141526,0.0
2,0.228622,0.000229,0.000317,1136.471,1031.228,2.0,180.0,-71.945084,19.489212,0.0,-19.350,-0.0,0.139981,0.0
3,0.193133,0.000223,0.000293,1136.471,1031.228,3.0,180.0,-71.945084,19.489212,0.0,-19.350,-0.0,0.133106,0.0
4,0.258888,0.000187,0.000269,1136.471,1031.228,4.0,180.0,-71.945084,19.489212,0.0,-19.350,-0.0,0.145066,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
332635,-0.000211,0.371312,0.360556,0.854,1.033,15.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.134427,0.0
332636,-0.018895,0.375383,0.358165,0.854,1.033,16.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.132863,0.0
332637,0.008087,0.326700,0.316900,0.854,1.033,17.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.135519,0.0
332638,0.012527,0.273002,0.262333,0.854,1.033,18.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.135568,0.0


In [5]:
big_df = pd.concat([df1, df2], ignore_index=True)
big_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut,shadows
0,0.187436,0.000161,0.000196,1124.69017,1020.232214,0.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.116122,0.0
1,0.092571,0.000277,0.000303,1124.69017,1020.232214,1.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.106717,0.0
2,0.057824,0.000337,0.000359,1124.69017,1020.232214,2.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.097078,0.0
3,0.171885,0.000224,0.000273,1124.69017,1020.232214,3.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.112769,0.0
4,0.110673,0.000253,0.000287,1124.69017,1020.232214,4.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.103993,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
662255,-0.000211,0.371312,0.360556,0.85400,1.033000,15.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.134427,0.0
662256,-0.018895,0.375383,0.358165,0.85400,1.033000,16.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.132863,0.0
662257,0.008087,0.326700,0.316900,0.85400,1.033000,17.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.135519,0.0
662258,0.012527,0.273002,0.262333,0.85400,1.033000,18.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0,0.135568,0.0


In [6]:
big_df.to_csv("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/shadows-detection-dataset/combined-dl-shadows.csv", index=False)